# 02 - Bronze to Silver
Casts the all-text Bronze columns to proper types, cleans them, and **MERGEs** the result into
`silver_earthquakes` (one row per event, latest revision). Rows that fail validation go to
`silver_quarantine` instead of crashing the run.

**Parameters** (widgets):
* `batch_id`   - only process Bronze rows from this batch (empty = all Bronze rows)
* `start_date` / `end_date` - only keep events whose event date is in this range, format `YYYY-MM-DD` (empty = no limit)

**Idempotent:** Silver key is `event_id`. A row is updated only when the incoming revision is newer
(`updated_time` greater), so re-running the same data changes nothing.
**Fault tolerant:** Bronze is processed one source file at a time, each with its own log row.

In [0]:
from datetime import datetime
import uuid

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType, LongType,
)

CATALOG, SCHEMA = "workspace", "seismic_pulse"
BRONZE = f"{CATALOG}.{SCHEMA}.bronze_earthquakes"
SILVER = f"{CATALOG}.{SCHEMA}.silver_earthquakes"
QUARANTINE = f"{CATALOG}.{SCHEMA}.silver_quarantine"
LOG_TABLE = f"{CATALOG}.{SCHEMA}.pipeline_execution_logs"

try:
    spark.conf.set("spark.sql.session.timeZone", "UTC")   # keep epoch conversion in UTC
except Exception:
    pass

dbutils.widgets.text("batch_id", "")
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")

## Contract: target type of every numeric column
Serverless Spark runs in ANSI mode, where a plain CAST can crash on bad text, so everything uses `try_cast`
(returns NULL instead). Integer columns go through double first so values like `"3.0"` still convert.

In [0]:
NUMERIC_TYPES = {
    "mag": "double", "time_ms": "bigint", "updated_ms": "bigint", "felt": "int",
    "cdi": "double", "mmi": "double", "tsunami": "int", "sig": "int", "nst": "int",
    "dmin": "double", "rms": "double", "gap": "double",
    "longitude": "double", "latitude": "double", "depth": "double",
}
REQUIRED = ["time_ms", "updated_ms", "latitude", "longitude"]   # a row without these is unusable


def add_typed_columns(df):
    for name, typ in NUMERIC_TYPES.items():
        if typ in ("int", "bigint"):
            expr = f"try_cast(try_cast({name} as double) as {typ})"
        else:
            expr = f"try_cast({name} as {typ})"
        df = df.withColumn(f"n_{name}", F.expr(expr))
    return df


def rejection_reason():
    """Text listing every problem found on the row (empty string = row is valid)."""
    checks = []
    for name in NUMERIC_TYPES:
        if name in REQUIRED:
            checks.append(F.when(F.col(f"n_{name}").isNull(), F.lit(f"MISSING_OR_BAD:{name}")))
        else:
            checks.append(F.when(F.col(name).isNotNull() & F.col(f"n_{name}").isNull(),
                                 F.lit(f"BAD_TYPE:{name}")))
    checks.append(F.when(~F.col("n_latitude").between(-90, 90), F.lit("LATITUDE_OUT_OF_RANGE")))
    checks.append(F.when(~F.col("n_longitude").between(-180, 180), F.lit("LONGITUDE_OUT_OF_RANGE")))
    return F.concat_ws(";", *checks)

## Silver transformation

In [0]:
MAG_TYPE_EXPR = """
CASE lower(trim(mag_type))
  WHEN 'ml' THEN 'ML'   WHEN 'md' THEN 'Md'   WHEN 'mb' THEN 'mb'  WHEN 'mb_lg' THEN 'mb_lg'
  WHEN 'ms' THEN 'Ms'   WHEN 'ms_20' THEN 'Ms'
  WHEN 'mw' THEN 'Mw'   WHEN 'mwr' THEN 'Mw'  WHEN 'mww' THEN 'Mw' WHEN 'mwc' THEN 'Mw' WHEN 'mwb' THEN 'Mw'
  WHEN 'mh' THEN 'Mh'   WHEN 'mi' THEN 'Mi'   WHEN 'mwp' THEN 'Mwp'
  ELSE coalesce(lower(trim(mag_type)), 'unknown')
END
"""

DEPTH_EXPR = """
CASE WHEN n_depth IS NULL THEN 'unknown'
     WHEN n_depth < 70 THEN 'shallow'
     WHEN n_depth < 300 THEN 'intermediate'
     ELSE 'deep' END
"""


def build_silver(valid):
    state = F.regexp_extract("place", r",\s*([A-Z]{2})$", 1)          # e.g. ", CA"
    last_part = F.trim(F.regexp_extract("place", r",\s*([^,]+)$", 1))  # text after the last comma
    region = (F.when(F.col("place").isNull(), F.lit("unknown"))
              .when(state != "", F.concat(F.lit("USA ("), state, F.lit(")")))
              .when(last_part != "", last_part)
              .otherwise(F.trim("place")))

    return valid.select(
        "event_id",
        F.expr("timestamp_millis(n_time_ms)").alias("event_time"),
        F.expr("to_date(timestamp_millis(n_time_ms))").alias("event_date"),
        F.expr("hour(timestamp_millis(n_time_ms))").alias("event_hour"),
        F.expr("timestamp_millis(n_updated_ms)").alias("updated_time"),
        F.col("n_mag").alias("magnitude"),
        F.expr(MAG_TYPE_EXPR).alias("mag_type"),
        F.col("n_latitude").alias("latitude"),
        F.col("n_longitude").alias("longitude"),
        F.col("n_depth").alias("depth_km"),
        F.expr(DEPTH_EXPR).alias("depth_category"),
        "place",
        region.alias("region"),
        F.coalesce(F.lower(F.trim("status")), F.lit("unknown")).alias("review_status"),
        F.col("n_sig").alias("significance"),
        F.col("n_felt").alias("felt"),
        F.col("n_cdi").alias("cdi"),
        F.col("n_mmi").alias("mmi"),
        "alert",
        F.col("n_tsunami").alias("tsunami"),
        "net",
        "event_type",
        F.col("n_nst").alias("nst"),
        F.col("n_gap").alias("gap"),
        F.col("n_rms").alias("rms"),
        F.col("n_dmin").alias("dmin"),
        "title",
        "batch_id",
        F.current_timestamp().alias("load_timestamp"),
    )

## Logging helper

In [0]:
LOG_SCHEMA = StructType([
    StructField("run_id", StringType()), StructField("layer", StringType()),
    StructField("load_type", StringType()), StructField("parameter", StringType()),
    StructField("start_time", TimestampType()), StructField("end_time", TimestampType()),
    StructField("status", StringType()), StructField("rows_inserted", LongType()),
    StructField("rows_updated", LongType()), StructField("rows_quarantined", LongType()),
    StructField("error_message", StringType()),
])


def log_run(run_id, load_type, parameter, start, end, status,
            inserted=0, updated=0, quarantined=0, error=None):
    row = [(run_id, "Bronze-to-Silver", load_type, parameter, start, end, status,
            int(inserted), int(updated), int(quarantined), error)]
    spark.createDataFrame(row, LOG_SCHEMA).write.mode("append").insertInto(LOG_TABLE)

## Process one Bronze source file

In [0]:
def process_chunk(bronze_df, batch_id, start_date, end_date):
    """Type, validate, split into good/bad rows, MERGE both. Returns (inserted, updated, quarantined)."""
    original_cols = bronze_df.columns
    typed = add_typed_columns(bronze_df).withColumn("reject_reason", rejection_reason())

    # ---- bad rows -> quarantine (idempotent MERGE on event_id + updated_ms)
    bad = (typed.filter(F.col("reject_reason") != "")
           .select("event_id", "updated_ms", "reject_reason",
                   F.to_json(F.struct(*[F.col(c) for c in original_cols if c != "extra_properties"])).alias("raw_record"),
                   "batch_id", F.current_timestamp().alias("load_timestamp"))
           .dropDuplicates(["event_id", "updated_ms"]))
    bad.createOrReplaceTempView("quarantine_src")
    q = spark.sql(f"""
        MERGE INTO {QUARANTINE} AS t
        USING quarantine_src AS s
        ON t.event_id = s.event_id AND t.updated_ms <=> s.updated_ms
        WHEN NOT MATCHED THEN INSERT *
    """).first()
    quarantined = q["num_inserted_rows"]

    # ---- good rows -> Silver: keep only the latest revision of each event in this chunk
    good = typed.filter(F.col("reject_reason") == "")
    w = Window.partitionBy("event_id").orderBy(F.col("n_updated_ms").desc(), F.col("load_timestamp").desc())
    latest = good.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")

    silver = build_silver(latest)
    if start_date:
        silver = silver.filter(F.col("event_date") >= F.lit(start_date).cast("date"))
    if end_date:
        silver = silver.filter(F.col("event_date") <= F.lit(end_date).cast("date"))

    silver.createOrReplaceTempView("silver_src")
    m = spark.sql(f"""
        MERGE INTO {SILVER} AS t
        USING silver_src AS s
        ON t.event_id = s.event_id
        WHEN MATCHED AND s.updated_time > t.updated_time THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """).first()
    return m["num_inserted_rows"], m["num_updated_rows"], quarantined

## Run for the given parameters

In [0]:
def run_bronze_to_silver(batch_id="", start_date="", end_date=""):
    run_id = str(uuid.uuid4())
    bronze = spark.table(BRONZE)
    if batch_id:
        bronze = bronze.filter(F.col("batch_id") == batch_id)

    chunks = [(r["source_file"], r["load_type"])
              for r in bronze.select("source_file", "load_type").distinct().orderBy("source_file").collect()]
    print(f"batch_id='{batch_id}' | dates '{start_date}'..'{end_date}' | source files to process: {len(chunks)}")

    ok, failed, ins_total, upd_total, quar_total = 0, 0, 0, 0, 0
    for path, load_type in chunks:
        start = datetime.now()
        param = f"{path.split('/')[-1]} | dates {start_date or '-'}..{end_date or '-'}"
        try:
            chunk_df = bronze.filter((F.col("source_file") == path) & (F.col("load_type") == load_type))
            ins, upd, quar = process_chunk(chunk_df, batch_id, start_date, end_date)
            log_run(run_id, load_type, param, start, datetime.now(), "Success", ins, upd, quar)
            ok += 1
            ins_total += ins; upd_total += upd; quar_total += quar
            print(f"OK   {path.split('/')[-1]}: {ins} inserted, {upd} updated, {quar} quarantined")
        except Exception as e:
            log_run(run_id, load_type, param, start, datetime.now(), "Failure", error=str(e)[:1000])
            failed += 1
            print(f"FAIL {path.split('/')[-1]}: {str(e)[:200]}")

    print(f"\nDone. Success: {ok} | Failed: {failed} | Inserted: {ins_total} | Updated: {upd_total} | Quarantined: {quar_total}")


run_bronze_to_silver(
    dbutils.widgets.get("batch_id"),
    dbutils.widgets.get("start_date"),
    dbutils.widgets.get("end_date"),
)

batch_id='' | dates ''..'' | source files to process: 40
OK   drift_test.geojson: 3 inserted, 0 updated, 2 quarantined
OK   usgs_2015-01-01.geojson: 2220 inserted, 0 updated, 0 quarantined
OK   usgs_2015-02-01.geojson: 1946 inserted, 0 updated, 0 quarantined
OK   usgs_2015-03-01.geojson: 2116 inserted, 0 updated, 0 quarantined
OK   usgs_2015-04-01.geojson: 2012 inserted, 0 updated, 0 quarantined
OK   usgs_2015-05-01.geojson: 2439 inserted, 0 updated, 0 quarantined
OK   usgs_2015-06-01.geojson: 1948 inserted, 0 updated, 0 quarantined
OK   usgs_2015-07-01.geojson: 2573 inserted, 0 updated, 0 quarantined
OK   usgs_2015-08-01.geojson: 2285 inserted, 0 updated, 0 quarantined
OK   usgs_2015-09-01.geojson: 2975 inserted, 0 updated, 0 quarantined
OK   usgs_2015-10-01.geojson: 2138 inserted, 0 updated, 0 quarantined
OK   usgs_2015-11-01.geojson: 2341 inserted, 0 updated, 0 quarantined
OK   usgs_2015-12-01.geojson: 2061 inserted, 0 updated, 0 quarantined
OK   usgs_2016-01-01.geojson: 2290 insert

## Checks (look at these after every run)

In [0]:
s = spark.table(SILVER)
print("Silver rows:", s.count(), "| distinct event_id:", s.select("event_id").distinct().count())
print("Quarantine rows:", spark.table(QUARANTINE).count())
s.printSchema()
display(spark.table(LOG_TABLE).filter("layer = 'Bronze-to-Silver'").orderBy(F.col("start_time").desc()).limit(10))
display(s.groupBy("depth_category").count())
display(s.groupBy("mag_type").count().orderBy(F.col("count").desc()))
display(s.limit(5))

Silver rows: 79634 | distinct event_id: 79634
Quarantine rows: 2
root
 |-- event_id: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- event_date: date (nullable = true)
 |-- event_hour: integer (nullable = true)
 |-- updated_time: timestamp (nullable = true)
 |-- magnitude: double (nullable = true)
 |-- mag_type: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- depth_km: double (nullable = true)
 |-- depth_category: string (nullable = true)
 |-- place: string (nullable = true)
 |-- region: string (nullable = true)
 |-- review_status: string (nullable = true)
 |-- significance: integer (nullable = true)
 |-- felt: integer (nullable = true)
 |-- cdi: double (nullable = true)
 |-- mmi: double (nullable = true)
 |-- alert: string (nullable = true)
 |-- tsunami: integer (nullable = true)
 |-- net: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- nst: integer (nullable = true)
 |-- 

run_id,layer,load_type,parameter,start_time,end_time,status,rows_inserted,rows_updated,rows_quarantined,error_message
22dc1831-55a6-4c15-97cc-637525a00435,Bronze-to-Silver,incremental,incremental_load_sample.geojson | dates -..-,2026-10-10T13:14:37.396Z,2026-10-10T13:14:46.579Z,Success,0,0,0,null
22dc1831-55a6-4c15-97cc-637525a00435,Bronze-to-Silver,full,usgs_2024-02-01.geojson | dates -..-,2026-10-10T13:14:27.362Z,2026-10-10T13:14:36.129Z,Success,0,0,0,null
22dc1831-55a6-4c15-97cc-637525a00435,Bronze-to-Silver,full,usgs_2024-01-01.geojson | dates -..-,2026-10-10T13:14:17.417Z,2026-10-10T13:14:26.203Z,Success,0,0,0,null
22dc1831-55a6-4c15-97cc-637525a00435,Bronze-to-Silver,full,usgs_2017-12-01.geojson | dates -..-,2026-10-10T13:14:08.252Z,2026-10-10T13:14:16.082Z,Success,1682,0,0,null
22dc1831-55a6-4c15-97cc-637525a00435,Bronze-to-Silver,full,usgs_2017-11-01.geojson | dates -..-,2026-10-10T13:13:58.761Z,2026-10-10T13:14:05.760Z,Success,1896,0,0,null
22dc1831-55a6-4c15-97cc-637525a00435,Bronze-to-Silver,full,usgs_2017-10-01.geojson | dates -..-,2026-10-10T13:13:50.153Z,2026-10-10T13:13:57.614Z,Success,1606,0,0,null
22dc1831-55a6-4c15-97cc-637525a00435,Bronze-to-Silver,full,usgs_2017-09-01.geojson | dates -..-,2026-10-10T13:13:41.782Z,2026-10-10T13:13:48.980Z,Success,2505,0,0,null
22dc1831-55a6-4c15-97cc-637525a00435,Bronze-to-Silver,full,usgs_2017-08-01.geojson | dates -..-,2026-10-10T13:13:32.361Z,2026-10-10T13:13:40.553Z,Success,1731,0,0,null
22dc1831-55a6-4c15-97cc-637525a00435,Bronze-to-Silver,full,usgs_2017-07-01.geojson | dates -..-,2026-10-10T13:13:24.074Z,2026-10-10T13:13:31.190Z,Success,2020,0,0,null
22dc1831-55a6-4c15-97cc-637525a00435,Bronze-to-Silver,full,usgs_2017-06-01.geojson | dates -..-,2026-10-10T13:13:15.209Z,2026-10-10T13:13:22.691Z,Success,1713,0,0,null


depth_category,count
shallow,61072
deep,3800
intermediate,14762


mag_type,count
mb,42859
ML,22915
Md,7911
Mw,5118
mb_lg,766
Mh,31
mc,20
mlg,9
mlr,3
unknown,1


event_id,event_time,event_date,event_hour,updated_time,magnitude,mag_type,latitude,longitude,depth_km,depth_category,place,region,review_status,significance,felt,cdi,mmi,alert,tsunami,net,event_type,nst,gap,rms,dmin,title,batch_id,load_timestamp
ak02410h1tlw,2024-01-22T05:42:12.294Z,2024-01-22,5,2024-03-30T22:32:18.040Z,3.1,ML,61.9481,-149.5529,55.9,shallow,"28 km NW of Fishhook, Alaska",Alaska,reviewed,150,10,2.5,3.022,null,0,ak,earthquake,null,null,0.71,null,"M 3.1 - 28 km NW of Fishhook, Alaska",batch_20261010_084253,2026-10-10T08:56:44.690Z
ak02410mid1x,2024-01-22T14:59:06.299Z,2024-01-22,14,2024-03-30T22:32:22.040Z,2.5,ML,64.4798,-146.9546,10.2,shallow,"5 km SSW of Salcha, Alaska",Alaska,reviewed,97,1,3.8,null,null,0,ak,earthquake,null,null,0.71,null,"M 2.5 - 5 km SSW of Salcha, Alaska",batch_20261010_084253,2026-10-10T08:56:44.690Z
ak02410p5dky,2024-01-22T19:02:36.310Z,2024-01-22,19,2024-03-30T22:32:27.040Z,3.5,ML,60.2082,-141.1444,16.2,shallow,"108 km NW of Yakutat, Alaska",Alaska,reviewed,188,null,null,3.761,null,0,ak,earthquake,null,null,0.69,null,"M 3.5 - 108 km NW of Yakutat, Alaska",batch_20261010_084253,2026-10-10T08:56:44.690Z
ak02410p7x32,2024-01-22T19:14:23.126Z,2024-01-22,19,2024-03-30T22:32:27.040Z,2.8,ML,57.7575,-156.724,131.0,intermediate,"48 km NE of Ugashik, Alaska",Alaska,reviewed,121,null,null,null,null,0,ak,earthquake,null,null,0.63,null,"M 2.8 - 48 km NE of Ugashik, Alaska",batch_20261010_084253,2026-10-10T08:56:44.690Z
ak02410rjaql,2024-01-22T23:03:22.283Z,2024-01-22,23,2024-03-30T22:32:33.040Z,2.5,ML,62.1522,-152.6167,0.3,shallow,"66 km WNW of Skwentna, Alaska",Alaska,reviewed,96,null,null,null,null,0,ak,earthquake,null,null,0.6,null,"M 2.5 - 66 km WNW of Skwentna, Alaska",batch_20261010_084253,2026-10-10T08:56:44.690Z


In [0]:
run_bronze_to_silver("incr_demo", "", "")    # batch_id, start_date, end_date

batch_id='incr_demo' | dates ''..'' | source files to process: 1
OK   incremental_load_sample.geojson: 0 inserted, 0 updated, 0 quarantined

Done. Success: 1 | Failed: 0 | Inserted: 0 | Updated: 0 | Quarantined: 0


In [0]:
run_bronze_to_silver("drift_test_1", "", "")

batch_id='drift_test_1' | dates ''..'' | source files to process: 1
OK   drift_test.geojson: 0 inserted, 0 updated, 0 quarantined

Done. Success: 1 | Failed: 0 | Inserted: 0 | Updated: 0 | Quarantined: 0


In [0]:
display(spark.sql("SELECT event_id, magnitude, latitude, review_status FROM workspace.seismic_pulse.silver_earthquakes WHERE event_id LIKE 'test_%'"))
display(spark.sql("SELECT event_id, reject_reason FROM workspace.seismic_pulse.silver_quarantine"))
display(spark.sql("SELECT drift_type, column_name, detail FROM workspace.seismic_pulse.schema_drift_log WHERE batch_id = 'drift_test_1'"))
display(spark.sql("SELECT layer, parameter, status, rows_inserted, rows_updated, rows_quarantined, start_time FROM workspace.seismic_pulse.pipeline_execution_logs ORDER BY start_time DESC LIMIT 10"))

event_id,magnitude,latitude,review_status
test_0_us6000u18k,7.7,7.5868,reviewed
test_4_aka2026uagrtn,2.5,62.313,automatic
test_5_aka2026uaglfz,2.6,60.168,automatic


event_id,reject_reason
test_2_us6000u18e,LATITUDE_OUT_OF_RANGE
test_1_aka2026uajubu,BAD_TYPE:mag


drift_type,column_name,detail
NULL_KEY,event_id,1 rows dropped
NEW_COLUMN,brand_new_field,stored in extra_properties
TYPE_DRIFT,mag,1 non-numeric values (will be quarantined in Silver)
NULL_KEY,event_id,1 rows dropped
NEW_COLUMN,brand_new_field,stored in extra_properties
TYPE_DRIFT,mag,1 non-numeric values (will be quarantined in Silver)


layer,parameter,status,rows_inserted,rows_updated,rows_quarantined,start_time
Bronze-to-Silver,drift_test.geojson | dates -..-,Success,0,0,0,2026-10-10T13:15:05.550Z
Bronze-to-Silver,incremental_load_sample.geojson | dates -..-,Success,0,0,0,2026-10-10T13:14:55.064Z
Bronze-to-Silver,incremental_load_sample.geojson | dates -..-,Success,0,0,0,2026-10-10T13:14:37.396Z
Bronze-to-Silver,usgs_2024-02-01.geojson | dates -..-,Success,0,0,0,2026-10-10T13:14:27.362Z
Bronze-to-Silver,usgs_2024-01-01.geojson | dates -..-,Success,0,0,0,2026-10-10T13:14:17.417Z
Bronze-to-Silver,usgs_2017-12-01.geojson | dates -..-,Success,1682,0,0,2026-10-10T13:14:08.252Z
Bronze-to-Silver,usgs_2017-11-01.geojson | dates -..-,Success,1896,0,0,2026-10-10T13:13:58.761Z
Bronze-to-Silver,usgs_2017-10-01.geojson | dates -..-,Success,1606,0,0,2026-10-10T13:13:50.153Z
Bronze-to-Silver,usgs_2017-09-01.geojson | dates -..-,Success,2505,0,0,2026-10-10T13:13:41.782Z
Bronze-to-Silver,usgs_2017-08-01.geojson | dates -..-,Success,1731,0,0,2026-10-10T13:13:32.361Z


In [0]:
for t in ["bronze_earthquakes", "silver_earthquakes", "silver_quarantine"]:
    spark.sql(f"DELETE FROM workspace.seismic_pulse.{t} WHERE event_id LIKE 'test_%'")
print("Test rows removed.")

Test rows removed.


In [0]:
run_bronze_to_silver("", "", "")

batch_id='' | dates ''..'' | source files to process: 39
OK   usgs_2015-01-01.geojson: 0 inserted, 0 updated, 0 quarantined
OK   usgs_2015-02-01.geojson: 0 inserted, 0 updated, 0 quarantined
OK   usgs_2015-03-01.geojson: 0 inserted, 0 updated, 0 quarantined
OK   usgs_2015-04-01.geojson: 0 inserted, 0 updated, 0 quarantined
OK   usgs_2015-05-01.geojson: 0 inserted, 0 updated, 0 quarantined
OK   usgs_2015-06-01.geojson: 0 inserted, 0 updated, 0 quarantined
OK   usgs_2015-07-01.geojson: 0 inserted, 0 updated, 0 quarantined
OK   usgs_2015-08-01.geojson: 0 inserted, 0 updated, 0 quarantined
OK   usgs_2015-09-01.geojson: 0 inserted, 0 updated, 0 quarantined
OK   usgs_2015-10-01.geojson: 0 inserted, 0 updated, 0 quarantined
OK   usgs_2015-11-01.geojson: 0 inserted, 0 updated, 0 quarantined
OK   usgs_2015-12-01.geojson: 0 inserted, 0 updated, 0 quarantined
OK   usgs_2016-01-01.geojson: 0 inserted, 0 updated, 0 quarantined
OK   usgs_2016-02-01.geojson: 0 inserted, 0 updated, 0 quarantined
OK   

In [0]:
for t in ["bronze_earthquakes", "silver_earthquakes", "silver_quarantine", "pipeline_execution_logs", "schema_drift_log"]:
    print(t, spark.table(f"workspace.seismic_pulse.{t}").count())

bronze_earthquakes 79631
silver_earthquakes 79631
silver_quarantine 0
pipeline_execution_logs 157
schema_drift_log 6
